# OpenAire raw EDA
 
## Setup

In [1]:
import sys
sys.path.insert(0, '/home/lu72hip/DIGICHer/dh_pipeline/src')
import pandas as pd
from pathlib import Path
from lib.database.duck.create_connection import create_duck_connection
from utils.config.config_loader import get_query_config

config = get_query_config()['openaire_dump']
DB = Path(config["path_duck"])

con = create_duck_connection(str(DB))
con.execute("SET memory_limit='64GB'")
con.execute("SET threads=16")
print(f'Connected to {DB}')

pd.set_option("display.max_rows", 21) # Show more rows  # or None for unlimited
pd.set_option("display.max_columns", None) # Show more columns  # None = show all
# pd.set_option("display.max_colwidth", None) # Don't truncate column content  # None = full content
# pd.set_option("display.width", None) # Wider display  # Auto-detect terminal width

Connected to /vast/lu72hip/data/duckdb/sources/openaire_raw.duckdb


## General Stats

In [5]:
### Get counts of all tables

q = """ 
SELECT
    table_name,
    estimated_size as cnt
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY estimated_size DESC
"""
con.execute(q).df()

,table_name,cnt
0,relation,562206510
1,work,205841448
2,project,3673360
3,organization,448161


In [4]:
### Q1 — Confirm org coverage (baseline)

q = """ 
  SELECT                                                                                                                                                       
      COUNT(*)                                                        AS total_orgs,                                                                           
      COUNT(rorId)                                                    AS has_ror_id,                                                                           
      ROUND(COUNT(rorId) * 100.0 / COUNT(*), 1)                      AS ror_pct,                                                                               
      COUNT(countryCode)                                              AS has_country_code,                                                                     
      ROUND(COUNT(countryCode) * 100.0 / COUNT(*), 1)                AS country_pct                                                                            
  FROM organization;                                                                                                                                           
"""
con.execute(q).df()

,total_orgs,has_ror_id,ror_pct,has_country_code,country_pct
0,448161,123542,27.6,319664,71.3


In [5]:
### Q2 — Project-linked orgs breakdown

q = """ 
SELECT                                                                                                                                                       
    COUNT(DISTINCT o.id)                                            AS project_linked_total,                                                                 
    COUNT(DISTINCT o.id) FILTER (WHERE o.rorId IS NOT NULL)        AS has_ror,                                                                               
    COUNT(DISTINCT o.id) FILTER (WHERE o.countryCode IS NOT NULL)  AS has_country,                                                                           
    COUNT(DISTINCT o.id) FILTER (                                                                                                                            
        WHERE o.rorId IS NULL AND o.countryCode IS NOT NULL                                                                                                  
    )                                                               AS no_ror_but_has_country                                                                
FROM organization o                                                                                                                                          
JOIN relation r ON r.source = o.id                                                                                                                           
    AND r.sourceType = 'organization'                                                                                                                        
    AND r.targetType = 'project';        
"""
con.execute(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,project_linked_total,has_ror,has_country,no_ror_but_has_country
0,308537,17878,185872,168012


# ok


In [14]:
### Q_raw1 — What pid schemes actually exist in the org data?

q = """ 
SELECT scheme, COUNT(DISTINCT id) AS orgs
FROM (SELECT id, unnest(pids).scheme AS scheme FROM organization)                                                                                            
GROUP BY scheme                                                  
ORDER BY orgs DESC;              
"""
con.execute(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,scheme,orgs
0,ROR,123542
1,GRID,108191
2,PIC,69287
3,ISNI,59194
4,Wikidata,56164
5,mag_id,26448
6,FundRef,17919
7,OrgRef,15148
8,OrgReg,6477
9,RNSR,6412


In [15]:
### Q_raw2 — GRID coverage on ALL orgs (the number we care about)

q = """ 

SELECT                                                                                                                                                       
    COUNT(*)                                                    AS total,
    COUNT(CASE WHEN list_filter(pids, p -> p.scheme = 'ROR')  != [] THEN 1 END) AS has_ror,                                                                  
    COUNT(CASE WHEN list_filter(pids, p -> p.scheme = 'GRID') != [] THEN 1 END) AS has_grid,                                                                 
    COUNT(CASE WHEN list_filter(pids, p -> p.scheme = 'ISNI') != [] THEN 1 END) AS has_isni                                                                  
FROM organization;          
"""
con.execute(q).df()

,total,has_ror,has_grid,has_isni
0,448161,123542,108191,59194


# OK2

In [8]:
###

q = """ 
ATTACH '/vast/lu72hip/data/duckdb/sources/openaire_staging.duckdb' AS s (READ_ONLY);
                                                                                                                                                            
SELECT          
    pid.scheme,
    COUNT(DISTINCT o.id)  AS project_orgs_with_scheme
FROM organization o                                  
JOIN s.organization so ON so.openaireId = o.id                                                                                                               
JOIN s.relation r      ON r.source = so.id    
    AND r.sourceType = 'organization'                                                                                                                        
    AND r.targetType = 'project'                                                                                                                             
CROSS JOIN UNNEST(o.pids) AS t(pid)
WHERE so.rorId IS NULL         -- only orgs that currently have NO geolocation                                                                               
GROUP BY pid.scheme                                                                                                                                          
ORDER BY project_orgs_with_scheme DESC;  
"""
con.execute(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,scheme,project_orgs_with_scheme
0,PIC,61607
1,RNSR,5010
2,OrgReg,464
3,mag_id,347
4,CVR,162
5,GRID,41
6,VIAF,19
7,ISNI,19
8,RRID,10
9,RingGold,8


In [ ]:
### Q_total — denominator: how many project-linked orgs have no geolocation at all

q = """ 
SELECT COUNT(DISTINCT r.source) AS projects_with_geo_org                                                                                                     
FROM relation r 
JOIN organization o ON o.id = r.target
WHERE r."sourceType" = 'project'
AND r."targetType" = 'organization'                                                                                                                        
AND o.geolocation IS NOT NULL;
"""
con.execute(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,projects_with_orgs
0,3372450
